## **LangChain** - _ChromaDB_

**ChromaDB** es una base de datos vectorial (vector database) de código abierto, diseñada específicamente para aplicaciones impulsadas por Inteligencia Artificial.

A diferencia de las bases de datos tradicionales que guardan datos en tablas o documentos y buscan por coincidencias exactas de palabras, **ChromaDB** guarda **embeddings**.
> Nota: **PostgreSQL** cuenta con extensiones (**pgvector**) que permiten guardar **embeddings**.

Ventajas de usar **ChromaDB**:

- **Búsqueda Semántica**: No busca palabras clave, busca por contexto y significado.

- **Ejecución Local**: Se ejecuta directamente en la memoria de la computadora usando **SQLite** como motor.

- **Simplicidad de Integración**: Con apenas unas líneas de código en **Python**, puedes crear una colección, guardar documentos y hacer búsquedas.

- **Modelos Incluidos**: Ya viene con modelos de **embeddings** por defecto (usando funciones de **HuggingFace** o **SentenceTransformers**).

In [ ]:
import numpy as np
import pandas as pd

from time import sleep

import chromadb

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

from dotenv import load_dotenv

load_dotenv()

EMBEDDING_MODEL = "gemini-embedding-2"
OUTPUT_DIMENSIONALITY = 3072

embeddings_model = GoogleGenerativeAIEmbeddings(model = EMBEDDING_MODEL, output_dimensionality = OUTPUT_DIMENSIONALITY)

### 01. Casos de Uso

- **Sistemas RAG** (_Retrieval-Augmented Generation_): Permite conectar PDFs, bases de conocimiento o manuales internos a un **LLM**. **ChromaDB** busca los párrafos relevantes y el **LLM** formula la respuesta final.

- **Buscadores Inteligentes**: Crear barras de búsqueda para e-commerce o wikis que entiendan la intención del usuario y toleren errores de ortografía o sinónimos.

- **Sistemas de Recomendación**: Sugerir artículos, películas o productos basados en qué tan "cerca" está un elemento de los intereses del usuario.

- **Memoria a Largo Plazo para Agentes**: Darle a un chatbot la capacidad de "recordar" conversaciones pasadas buscando en su historial de vectores.

### 02. Implementación Básica

In [ ]:
# Inicializamos el cliente
cliente = chromadb.Client()

# Creamos una colección
coleccion = cliente.create_collection(name = "documentos")

# Añadimos documentos a nuestra base de datos vectorial
# Como no le pasamos una función de embedding específica, 
# Chroma descargará y usará un modelo por defecto la primera vez.
coleccion.add(documents = ["El lenguaje Python fue creado por Guido van Rossum y es muy usado en ciencia de datos.",
                           "Para hacer una buena pizza napolitana necesitas un horno que alcance los 400 grados.",
                           "Chroma DB es una herramienta excelente para guardar y buscar embeddings de forma local."],
              metadatas = [{"tema" : "programacion"}, {"tema" : "cocina"}, {"tema" : "inteligencia artificial"}],
              ids = ["doc1", "doc2", "doc3"])

In [ ]:

# Hacemos una búsqueda semántica
resultados = coleccion.query(query_texts = ["¿Quién inventó ese lenguaje de programación famoso para datos?"],
                             n_results = 1)

print("Documento encontrado:", resultados['documents'][0][0])
print("Metadatos:", resultados['metadatas'][0][0])
print("Distancia:", resultados['distances'][0][0])

### 03. **Document**

En **LangChain**, el objeto `Document` es una de las estructuras de datos más fundamentales. Es una abstracción diseñada para representar un fragmento de texto junto con su información contextual.

Dado que los **LLMs** necesitan contexto para responder con precisión, **LangChain** utiliza este objeto para mover la información entre operaciones, desde la lectura de un archivo hasta su almacenamiento o envío al modelo.

El objeto `Document` está compuesto principalmente por dos atributos esenciales:

- `page_content` (_str_): Es el texto en sí. Puede ser el contenido completo de un artículo, un solo párrafo, una transcripción de un video o una fila de un archivo **.csv**. Es la información que el **LLM** finalmente leerá y procesará.

- `metadata` (_dict_): Almacena información arbitraria sobre el `page_content`. Es crucial para saber de dónde vino el texto, organizarlo o filtrarlo más adelante. Ejemplos típicos incluyen: la URL de origen, el nombre del archivo, el número de página, el autor o la fecha de creación.

In [ ]:
page_content = "LangChain es un framework para desarrollar aplicaciones impulsadas por modelos de lenguaje."
metadata = {"source" : "https://python.langchain.com/",
            "autor" : "Harrison Chase",
            "categoria" : "Inteligencia Artificial"}

documento = Document(page_content = page_content, metadata = metadata)

print(documento.page_content)

print(documento.metadata["source"])

### 04. Implementación Avanzada con **LangChain**

In [ ]:
# Documentos de ejemplo
documentos = [Document(page_content = "El lenguaje Python fue creado por Guido van Rossum y es ampliamente utilizado en Ciencia de Datos e IA.",
                       metadata = {"categoria": "programacion"}),
              Document(page_content = "Para cocinar una auténtica pizza napolitana se requiere un horno de leña que supere los 450 grados Celsius.",
                       metadata = {"categoria": "cocina"}),
              Document(page_content = "Los transformers son la arquitectura de red neuronal detrás de los modelos de lenguaje modernos como Gemini o GPT.",
                       metadata = {"categoria": "inteligencia artificial"})]

# Crear e indexar la base de datos vectorial Chroma
# LangChain se encarga de llamar a la API de Gemini para cada documento,
# obtener los vectores y guardarlos en Chroma en memoria.

vector_store = Chroma.from_documents(documents = documentos,
                                     embedding = embeddings_model,
                                     persist_directory = "../base_de_datos") # Guarda los datos en una carpeta

In [ ]:
# Realizar una búsqueda por similitud semántica
query = "¿Qué tecnología hace posibles los modelos de lenguaje actuales?"

# Buscamos los 2 documentos más similares
resultados = vector_store.similarity_search(query = query, k = 2)

for doc in resultados:
    print(f"Contenido: {doc.page_content}")
    print(f"Metadatos asociados: {doc.metadata}")

### 05. Caso de Uso: _Sistema de Recomendación_

In [ ]:
df = pd.read_csv(filepath_or_buffer = "../Data/todas_recetas.csv").sample(n = 20, random_state = 42)

df = df.replace({"[]" : np.nan})

df["categorias"] = df["categorias"].apply(lambda x : eval(x) if type(x) == str else np.nan)
df["ingredientes"] = df["ingredientes"].apply(lambda x : eval(x) if type(x) == str else np.nan)
df["instrucciones"] = df["instrucciones"].apply(lambda x : eval(x) if type(x) == str else np.nan)

df.head(3)

In [ ]:
documentos = list()

for i in range(df.shape[0]):

    fila = df.iloc[i, :]
    
    page_content = fila["descripciones"]
    metadata = fila.drop(["descripciones", "instrucciones"]).dropna().to_dict()

    documento = Document(page_content = page_content, metadata = metadata)

    documentos.append(documento)

    

In [ ]:
documentos

In [ ]:
for documento in documentos:

    vector_store = Chroma.from_documents(documents = [documento],
                                         embedding = embeddings_model,
                                         persist_directory = "../recetas")
    
    sleep(1)

In [ ]:
query = "Ensalada fresca para verano, 3 a 4 comensales, sin piña"

resultados = vector_store.similarity_search(query = query, k = 10)

for doc in resultados:
    print(f"Contenido: {doc.page_content}")
    print(f"Metadatos asociados: {doc.metadata}")